# Deep Learning Lab — Assignment 8
## Neural Language Translation: Attention Networks, Self-Attention and Multi-Head Attention

In [ ]:
# ── Global seed (must be first) ──────────────────────────────────────────────
import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

import re, math, copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Task 1: Data Preparation

In [ ]:
# ── 1-1  Load from TSV files (no header, columns: en, de) ────────────────────
def load_tsv(path):
    """Load a tab-separated file with two columns: English and German."""
    df = pd.read_csv(path, sep='\t', header=None, names=['en', 'de'],
                     quoting=3)   # quoting=3 → QUOTE_NONE (avoids quote issues)
    return df.to_dict('records')  # list of {'en': ..., 'de': ...}

train_data = load_tsv('train.tsv')
val_data   = load_tsv('val.tsv')
test_data  = load_tsv('test.tsv')

print('Train pairs  :', len(train_data))
print('Val pairs    :', len(val_data))
print('Test pairs   :', len(test_data))

# T01 — first test pair (index 0)
T01 = test_data[0]
print('\nT01 English :', T01['en'])
print('T01 German  :', T01['de'])

In [ ]:
# ── 1-2  Tokeniser ────────────────────────────────────────────────────────────
def tokenize(sentence: str):
    """Lowercase and split on word/punctuation boundaries."""
    sentence = sentence.lower()
    tokens = re.findall(r"\w+|[^\w\s]", sentence)
    return tokens

print(tokenize(T01['en']))
print(tokenize(T01['de']))

In [ ]:
# ── 1-3  Build vocabularies (training split only, min_freq=2) ─────────────────
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

def build_vocab(sentences, min_freq=2):
    counter = Counter()
    for s in sentences:
        counter.update(tokenize(s))
    stoi = {'<pad>': PAD_IDX, '<sos>': SOS_IDX, '<eos>': EOS_IDX, '<unk>': UNK_IDX}
    for token, freq in counter.items():
        if freq >= min_freq and token not in stoi:
            stoi[token] = len(stoi)
    itos = {v: k for k, v in stoi.items()}
    return stoi, itos

src_stoi, src_itos = build_vocab([ex['en'] for ex in train_data])
tgt_stoi, tgt_itos = build_vocab([ex['de'] for ex in train_data])

print('Source (EN) vocab size:', len(src_stoi))
print('Target (DE) vocab size:', len(tgt_stoi))

In [ ]:
# ── 1-4  Numericalize + pad/truncate to MAX_LEN=30 ───────────────────────────
MAX_LEN = 30

def numericalize(sentence, stoi, max_len=MAX_LEN):
    tokens = tokenize(sentence)
    ids = [SOS_IDX] + [stoi.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
    ids = ids[:max_len]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

# ── 1-5  Padding mask: True = valid token, False = <pad> ─────────────────────
def make_src_mask(ids):
    return [tok != PAD_IDX for tok in ids]

# ── 1-6  Display for T01 (source side) ───────────────────────────────────────
t01_en     = T01['en']
t01_tokens = tokenize(t01_en)
t01_ids    = numericalize(t01_en, src_stoi)
t01_mask   = make_src_mask(t01_ids)

print('Original     :', t01_en)
print('Tokens       :', t01_tokens)
print('Numericalized:', t01_ids)
print('Padding mask :', t01_mask)

In [ ]:
# ── 1-7  Build DataLoaders ────────────────────────────────────────────────────
BATCH_SIZE = 64

def build_tensors(split, src_stoi, tgt_stoi):
    src_ids, tgt_ids, src_masks = [], [], []
    for ex in split:
        s = numericalize(ex['en'], src_stoi)
        t = numericalize(ex['de'], tgt_stoi)
        m = make_src_mask(s)
        src_ids.append(s)
        tgt_ids.append(t)
        src_masks.append(m)
    return (
        torch.tensor(src_ids,   dtype=torch.long),
        torch.tensor(tgt_ids,   dtype=torch.long),
        torch.tensor(src_masks, dtype=torch.bool)
    )

train_src, train_tgt, train_mask = build_tensors(train_data, src_stoi, tgt_stoi)
val_src,   val_tgt,   val_mask   = build_tensors(val_data,   src_stoi, tgt_stoi)
test_src,  test_tgt,  test_mask  = build_tensors(test_data,  src_stoi, tgt_stoi)

train_loader = DataLoader(TensorDataset(train_src, train_tgt, train_mask),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(val_src,   val_tgt,   val_mask),
                          batch_size=BATCH_SIZE)
test_loader  = DataLoader(TensorDataset(test_src,  test_tgt,  test_mask),
                          batch_size=BATCH_SIZE)

print('Train batches:', len(train_loader))
print('Val batches  :', len(val_loader))
print('Test batches :', len(test_loader))

## Shared Model Components

In [ ]:
# ── Hyper-parameters ──────────────────────────────────────────────────────────
EMB_DIM   = 64
HID_DIM   = 128
N_EPOCHS  = 5
LR        = 1e-3
SRC_VOCAB = len(src_stoi)
TGT_VOCAB = len(tgt_stoi)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
# ── Encoder (shared across all tasks) ────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):                    # src: (B, S)
        embedded = self.embedding(src)         # (B, S, emb_dim)
        outputs, hidden = self.rnn(embedded)   # outputs: (B, S, hid_dim)
        return outputs, hidden                 # hidden: (1, B, hid_dim)

In [ ]:
# ── Generic training / evaluation helpers ─────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for src, tgt, mask in loader:
        src, tgt, mask = src.to(device), tgt.to(device), mask.to(device)
        optimizer.zero_grad()
        output = model(src, tgt, mask)   # (B, T, vocab)
        B, T, V = output.shape
        loss = criterion(output[:, :-1].reshape(-1, V),
                         tgt[:, 1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt, mask in loader:
            src, tgt, mask = src.to(device), tgt.to(device), mask.to(device)
            output = model(src, tgt, mask)
            B, T, V = output.shape
            loss = criterion(output[:, :-1].reshape(-1, V),
                             tgt[:, 1:].reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

def token_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for src, tgt, mask in loader:
            src, tgt, mask = src.to(device), tgt.to(device), mask.to(device)
            output = model(src, tgt, mask)      # (B, T, V)
            preds  = output[:, :-1].argmax(-1)  # (B, T-1)
            labels = tgt[:, 1:]                 # (B, T-1)
            non_pad = labels != PAD_IDX
            correct += (preds == labels)[non_pad].sum().item()
            total   += non_pad.sum().item()
    return correct / total * 100

def run_training(model, train_loader, val_loader, n_epochs, lr, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses = [], []
    for epoch in range(1, n_epochs + 1):
        tr = train_epoch(model, train_loader, optimizer, criterion, device)
        vl = evaluate(model, val_loader, criterion, device)
        train_losses.append(tr)
        val_losses.append(vl)
        print(f'Epoch {epoch}/{n_epochs}  Train Loss: {tr:.4f}  Val Loss: {vl:.4f}')
    return train_losses, val_losses

## Task 2: Additive (Bahdanau) Attention Seq2Seq

In [ ]:
# ── AdditiveAttention ─────────────────────────────────────────────────────────
class AdditiveAttention(nn.Module):
    """
    e_{t,i} = v^T tanh(W_h h_i + W_s s_t + b)
    """
    def __init__(self, hid_dim):
        super().__init__()
        self.Wh = nn.Linear(hid_dim, hid_dim, bias=False)
        self.Ws = nn.Linear(hid_dim, hid_dim, bias=True)   # b absorbed here
        self.v  = nn.Linear(hid_dim, 1,       bias=False)

    def forward(self, st, H, src_mask):
        """
        st      : (B, hid_dim)
        H       : (B, S, hid_dim)
        src_mask: (B, S) bool  — True = valid
        returns : context (B, hid_dim), alpha (B, S)
        """
        energy = torch.tanh(self.Wh(H) + self.Ws(st).unsqueeze(1))  # (B, S, H)
        scores = self.v(energy).squeeze(-1)                           # (B, S)
        scores = scores.masked_fill(~src_mask, float('-inf'))
        alpha  = torch.softmax(scores, dim=-1)                        # (B, S)
        context = torch.bmm(alpha.unsqueeze(1), H).squeeze(1)         # (B, H)
        return context, alpha


class DecoderAdditive(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = AdditiveAttention(hid_dim)
        self.gru       = nn.GRUCell(emb_dim + hid_dim, hid_dim)
        self.fc_out    = nn.Linear(hid_dim * 2 + emb_dim, vocab_size)

    def forward_step(self, token, hidden, enc_out, src_mask):
        emb = self.embedding(token)                           # (B, emb)
        context, alpha = self.attention(hidden, enc_out, src_mask)
        gru_in  = torch.cat([emb, context], dim=-1)
        hidden  = self.gru(gru_in, hidden)
        pred    = self.fc_out(torch.cat([hidden, context, emb], dim=-1))
        return pred, hidden, alpha


class Seq2SeqAdditive(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, src_mask, teacher_forcing_ratio=0.5):
        B, T = tgt.shape
        enc_out, hidden = self.encoder(src)
        hidden  = hidden.squeeze(0)
        outputs = torch.zeros(B, T, TGT_VOCAB).to(self.device)
        token   = tgt[:, 0]
        for t in range(1, T):
            pred, hidden, _ = self.decoder.forward_step(token, hidden, enc_out, src_mask)
            outputs[:, t]   = pred
            use_tf = random.random() < teacher_forcing_ratio
            token  = tgt[:, t] if use_tf else pred.argmax(-1)
        return outputs

In [ ]:
# ── Build & train Task-2 model ────────────────────────────────────────────────
random.seed(42); np.random.seed(42); torch.manual_seed(42)

enc2   = Encoder(SRC_VOCAB, EMB_DIM, HID_DIM).to(DEVICE)
dec2   = DecoderAdditive(TGT_VOCAB, EMB_DIM, HID_DIM).to(DEVICE)
model2 = Seq2SeqAdditive(enc2, dec2, DEVICE).to(DEVICE)

print('Task-2 parameters:', sum(p.numel() for p in model2.parameters()))
train_losses2, val_losses2 = run_training(
    model2, train_loader, val_loader, N_EPOCHS, LR, DEVICE)

In [ ]:
# ── Loss plot ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.plot(range(1, N_EPOCHS+1), train_losses2, 'o-',  label='Train')
plt.plot(range(1, N_EPOCHS+1), val_losses2,   's--', label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Task 2 — Additive Attention: Loss Curves')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Verify attention weights sum to 1 ────────────────────────────────────────
def verify_attn_sums_additive(model, loader, device, n_batches=5):
    model.eval()
    row_sums = []
    with torch.no_grad():
        for i, (src, tgt, mask) in enumerate(loader):
            if i >= n_batches: break
            src, tgt, mask = src.to(device), tgt.to(device), mask.to(device)
            enc_out, hidden = model.encoder(src)
            hidden = hidden.squeeze(0)
            token  = tgt[:, 0]
            for t in range(1, tgt.shape[1]):
                _, hidden, alpha = model.decoder.forward_step(
                    token, hidden, enc_out, mask)
                row_sums.append(alpha.sum(dim=-1).cpu())
                token = tgt[:, t]
    row_sums = torch.cat(row_sums)
    print(f'Min row sum: {row_sums.min():.6f}')
    print(f'Max row sum: {row_sums.max():.6f}')

verify_attn_sums_additive(model2, val_loader, DEVICE)

test_loss2 = evaluate(model2, test_loader, criterion, DEVICE)
test_acc2  = token_accuracy(model2, test_loader, DEVICE)
print(f'\nTask-2  Test Loss: {test_loss2:.4f}  Token Accuracy: {test_acc2:.2f}%')

## Task 3: Visualising Word–Word Relationships

In [ ]:
# ── Helper: extract attention matrix & words ──────────────────────────────────
def get_attn_additive(model, src_tensor, tgt_tensor, mask_tensor, device):
    model.eval()
    with torch.no_grad():
        src  = src_tensor.unsqueeze(0).to(device)
        tgt  = tgt_tensor.unsqueeze(0).to(device)
        mask = mask_tensor.unsqueeze(0).to(device)
        enc_out, hidden = model.encoder(src)
        hidden = hidden.squeeze(0)
        token  = tgt[:, 0]
        alphas = []
        for t in range(1, tgt.shape[1]):
            _, hidden, alpha = model.decoder.forward_step(
                token, hidden, enc_out, mask)
            alphas.append(alpha.squeeze(0).cpu().numpy())
            token = tgt[:, t]

    src_ids   = src_tensor.tolist()
    src_len   = sum(1 for x in src_ids if x != PAD_IDX)
    src_words = [src_itos[i] for i in src_ids[:src_len]]

    tgt_ids   = tgt_tensor.tolist()
    tgt_words = []
    for i in tgt_ids[1:]:
        w = tgt_itos.get(i, '<unk>')
        if w in ('<pad>', '<eos>'): break
        tgt_words.append(w)
    T_valid = len(tgt_words)
    alpha_mat = np.stack(alphas[:T_valid])[:, :src_len]
    return alpha_mat, src_words, tgt_words


def plot_heatmap(alpha, src_words, tgt_words, title):
    fig, ax = plt.subplots(figsize=(max(6, len(src_words)*0.65),
                                    max(4, len(tgt_words)*0.55)))
    im = ax.imshow(alpha, aspect='auto', cmap='Blues')
    ax.set_xticks(range(len(src_words)))
    ax.set_xticklabels(src_words, rotation=45, ha='right')
    ax.set_yticks(range(len(tgt_words)))
    ax.set_yticklabels(tgt_words)
    ax.set_xlabel('Source (EN)'); ax.set_ylabel('Target (DE)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Heatmaps + analysis for T01, T02, T03 ────────────────────────────────────
import numpy as np

for idx, label in [(0, 'T01'), (1, 'T02'), (2, 'T03')]:
    alpha, sw, tw = get_attn_additive(
        model2, test_src[idx], test_tgt[idx], test_mask[idx], DEVICE)
    plot_heatmap(alpha, sw, tw, f'Task 3 — Additive Attention: {label}')

    flat = [(alpha[t, s], tw[t], sw[s])
            for t in range(len(tw)) for s in range(len(sw))]
    flat.sort(reverse=True)
    print(f'{label} — two strongest alignments:')
    for weight, tword, sword in flat[:2]:
        print(f'  {tword} <- {sword}  alpha = {weight:.4f}')
    print()

## Task 4: Scaled Dot-Product Attention

In [ ]:
# ── ScaledDotProductAttention ─────────────────────────────────────────────────
class ScaledDotProductAttention(nn.Module):
    """
    Q = Xq W_Q,  K = Xk W_K,  V = Xk W_V
    S = Q K^T / sqrt(dk),  A = softmax(S),  O = A V
    """
    def __init__(self, hid_dim):
        super().__init__()
        self.W_Q = nn.Linear(hid_dim, hid_dim, bias=False)
        self.W_K = nn.Linear(hid_dim, hid_dim, bias=False)
        self.W_V = nn.Linear(hid_dim, hid_dim, bias=False)
        self.scale = math.sqrt(hid_dim)

    def forward(self, Xq, Xk, mask=None):
        """
        Xq  : (B, T_q, d)
        Xk  : (B, T_k, d)
        mask: (B, T_k) bool  — True = valid
        returns O (B, T_q, d), A (B, T_q, T_k)
        """
        Q = self.W_Q(Xq)
        K = self.W_K(Xk)
        V = self.W_V(Xk)
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale   # (B, T_q, T_k)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(1), float('-inf'))
        A = torch.softmax(scores, dim=-1)
        O = torch.bmm(A, V)
        return O, A


class DecoderScaled(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = ScaledDotProductAttention(hid_dim)
        self.gru       = nn.GRUCell(emb_dim + hid_dim, hid_dim)
        self.fc_out    = nn.Linear(hid_dim * 2 + emb_dim, vocab_size)

    def forward_step(self, token, hidden, enc_out, src_mask):
        emb = self.embedding(token)                         # (B, emb)
        context, alpha = self.attention(
            hidden.unsqueeze(1), enc_out, src_mask)         # (B,1,H)
        context = context.squeeze(1)                        # (B, H)
        alpha   = alpha.squeeze(1)                          # (B, S)
        gru_in  = torch.cat([emb, context], dim=-1)
        hidden  = self.gru(gru_in, hidden)
        pred    = self.fc_out(torch.cat([hidden, context, emb], dim=-1))
        return pred, hidden, alpha


class Seq2SeqScaled(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, src_mask, teacher_forcing_ratio=0.5):
        B, T = tgt.shape
        enc_out, hidden = self.encoder(src)
        hidden  = hidden.squeeze(0)
        outputs = torch.zeros(B, T, TGT_VOCAB).to(self.device)
        token   = tgt[:, 0]
        for t in range(1, T):
            pred, hidden, _ = self.decoder.forward_step(
                token, hidden, enc_out, src_mask)
            outputs[:, t] = pred
            use_tf = random.random() < teacher_forcing_ratio
            token  = tgt[:, t] if use_tf else pred.argmax(-1)
        return outputs

In [ ]:
# ── Build & train Task-4 model ────────────────────────────────────────────────
random.seed(42); np.random.seed(42); torch.manual_seed(42)

enc4   = Encoder(SRC_VOCAB, EMB_DIM, HID_DIM).to(DEVICE)
dec4   = DecoderScaled(TGT_VOCAB, EMB_DIM, HID_DIM).to(DEVICE)
model4 = Seq2SeqScaled(enc4, dec4, DEVICE).to(DEVICE)

print('Task-4 parameters:', sum(p.numel() for p in model4.parameters()))
train_losses4, val_losses4 = run_training(
    model4, train_loader, val_loader, N_EPOCHS, LR, DEVICE)

In [ ]:
# ── Loss plot ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.plot(range(1, N_EPOCHS+1), train_losses4, 'o-',  label='Train')
plt.plot(range(1, N_EPOCHS+1), val_losses4,   's--', label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Task 4 — Scaled Dot-Product Attention: Loss Curves')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Verify attention row sums ─────────────────────────────────────────────────
def verify_attn_sums_scaled(model, loader, device, n_batches=5):
    model.eval()
    row_sums = []
    with torch.no_grad():
        for i, (src, tgt, mask) in enumerate(loader):
            if i >= n_batches: break
            src, tgt, mask = src.to(device), tgt.to(device), mask.to(device)
            enc_out, hidden = model.encoder(src)
            hidden = hidden.squeeze(0)
            token  = tgt[:, 0]
            for t in range(1, tgt.shape[1]):
                _, hidden, alpha = model.decoder.forward_step(
                    token, hidden, enc_out, mask)
                row_sums.append(alpha.sum(dim=-1).cpu())
                token = tgt[:, t]
    row_sums = torch.cat(row_sums)
    print(f'Min row sum: {row_sums.min():.6f}')
    print(f'Max row sum: {row_sums.max():.6f}')

verify_attn_sums_scaled(model4, val_loader, DEVICE)

In [ ]:
# ── Cross-attention heatmaps for T04 and T05 ─────────────────────────────────
def get_attn_scaled(model, src_tensor, tgt_tensor, mask_tensor, device):
    model.eval()
    with torch.no_grad():
        src  = src_tensor.unsqueeze(0).to(device)
        tgt  = tgt_tensor.unsqueeze(0).to(device)
        mask = mask_tensor.unsqueeze(0).to(device)
        enc_out, hidden = model.encoder(src)
        hidden = hidden.squeeze(0)
        token  = tgt[:, 0]
        alphas = []
        for t in range(1, tgt.shape[1]):
            _, hidden, alpha = model.decoder.forward_step(
                token, hidden, enc_out, mask)
            alphas.append(alpha.squeeze(0).cpu().numpy())
            token = tgt[:, t]

    src_ids   = src_tensor.tolist()
    src_len   = sum(1 for x in src_ids if x != PAD_IDX)
    src_words = [src_itos[i] for i in src_ids[:src_len]]

    tgt_ids   = tgt_tensor.tolist()
    tgt_words = []
    for i in tgt_ids[1:]:
        w = tgt_itos.get(i, '<unk>')
        if w in ('<pad>', '<eos>'): break
        tgt_words.append(w)
    T_valid   = len(tgt_words)
    alpha_mat = np.stack(alphas[:T_valid])[:, :src_len]
    return alpha_mat, src_words, tgt_words


for idx, label in [(3, 'T04'), (4, 'T05')]:
    alpha, sw, tw = get_attn_scaled(
        model4, test_src[idx], test_tgt[idx], test_mask[idx], DEVICE)
    plot_heatmap(alpha, sw, tw,
        f'Task 4 — Scaled Dot-Product Attention: {label}')
    flat = [(alpha[t, s], tw[t], sw[s])
            for t in range(len(tw)) for s in range(len(sw))]
    flat.sort(reverse=True)
    wt, tword, sword = flat[0]
    print(f'{label}: strongest alignment  {tword} <- {sword}  alpha = {wt:.4f}\n')

In [ ]:
# ── Greedy decode helper (Task 4) ─────────────────────────────────────────────
def greedy_decode_scaled(model, src_tensor, mask_tensor, device, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src  = src_tensor.unsqueeze(0).to(device)
        mask = mask_tensor.unsqueeze(0).to(device)
        enc_out, hidden = model.encoder(src)
        hidden = hidden.squeeze(0)
        token  = torch.tensor([SOS_IDX], device=device)
        decoded = []
        for _ in range(max_len):
            pred, hidden, _ = model.decoder.forward_step(
                token, hidden, enc_out, mask)
            next_tok = pred.argmax(-1)
            word = tgt_itos.get(next_tok.item(), '<unk>')
            if word == '<eos>': break
            if word not in ('<pad>', '<sos>'):
                decoded.append(word)
            token = next_tok
    return ' '.join(decoded)


def get_sentence(src_tensor, tgt_tensor, src_itos, tgt_itos):
    src_ids  = src_tensor.tolist()
    src_len  = sum(1 for x in src_ids if x != PAD_IDX)
    src_sent = ' '.join(src_itos[i] for i in src_ids[:src_len]
                        if src_itos[i] not in ('<sos>', '<eos>', '<pad>'))
    tgt_ids  = tgt_tensor.tolist()
    tgt_words = []
    for i in tgt_ids[1:]:
        w = tgt_itos.get(i, '<unk>')
        if w in ('<pad>', '<eos>'): break
        tgt_words.append(w)
    return src_sent, ' '.join(tgt_words)


print('Task 4 — Greedy translations for T06, T07, T08')
for idx, label in [(5, 'T06'), (6, 'T07'), (7, 'T08')]:
    src_sent, tgt_sent = get_sentence(
        test_src[idx], test_tgt[idx], src_itos, tgt_itos)
    pred = greedy_decode_scaled(
        model4, test_src[idx], test_mask[idx], DEVICE)
    print(f'\n{label}')
    print(f'  English source    : {src_sent}')
    print(f'  Ground-truth DE   : {tgt_sent}')
    print(f'  Task-4 prediction : {pred}')

test_loss4 = evaluate(model4, test_loader, criterion, DEVICE)
test_acc4  = token_accuracy(model4, test_loader, DEVICE)
print(f'\nTask-4  Test Loss: {test_loss4:.4f}  Token Accuracy: {test_acc4:.2f}%')

## Task 5: Multi-Head Attention

In [ ]:
N_HEADS = 4
D_HEAD  = HID_DIM // N_HEADS   # 32

class MultiHeadAttention(nn.Module):
    """
    h=4 heads, d_model=128, d_head=32.
    Single Q/K/V projection, split into heads, per-head scaled dot-product,
    concatenate, apply W_O.
    """
    def __init__(self, hid_dim, n_heads):
        super().__init__()
        assert hid_dim % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = hid_dim // n_heads
        self.W_Q = nn.Linear(hid_dim, hid_dim, bias=False)
        self.W_K = nn.Linear(hid_dim, hid_dim, bias=False)
        self.W_V = nn.Linear(hid_dim, hid_dim, bias=False)
        self.W_O = nn.Linear(hid_dim, hid_dim, bias=False)
        self.scale = math.sqrt(self.d_head)

    def forward(self, Xq, Xk, mask=None):
        """
        Xq  : (B, T_q, d_model)
        Xk  : (B, T_k, d_model)
        mask: (B, T_k) bool
        returns O (B, T_q, d_model), A_list list[h] of (B, T_q, T_k)
        """
        B, T_q, _ = Xq.shape
        _, T_k, _ = Xk.shape
        h, d = self.n_heads, self.d_head

        Q = self.W_Q(Xq).view(B, T_q, h, d).transpose(1, 2)   # (B,h,T_q,d)
        K = self.W_K(Xk).view(B, T_k, h, d).transpose(1, 2)   # (B,h,T_k,d)
        V = self.W_V(Xk).view(B, T_k, h, d).transpose(1, 2)   # (B,h,T_k,d)

        scores = torch.matmul(Q, K.transpose(-1, -2)) / self.scale  # (B,h,T_q,T_k)
        if mask is not None:
            scores = scores.masked_fill(~mask[:, None, None, :], float('-inf'))
        A   = torch.softmax(scores, dim=-1)          # (B,h,T_q,T_k)
        O_h = torch.matmul(A, V)                     # (B,h,T_q,d_head)

        O_h = O_h.transpose(1, 2).contiguous().view(B, T_q, h * d)
        O   = self.W_O(O_h)                          # (B,T_q,d_model)

        A_list = [A[:, i] for i in range(h)]         # each (B,T_q,T_k)
        return O, A_list


class DecoderMHA(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_heads):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = MultiHeadAttention(hid_dim, n_heads)
        self.gru       = nn.GRUCell(emb_dim + hid_dim, hid_dim)
        self.fc_out    = nn.Linear(hid_dim * 2 + emb_dim, vocab_size)

    def forward_step(self, token, hidden, enc_out, src_mask):
        emb = self.embedding(token)
        context, A_list = self.attention(
            hidden.unsqueeze(1), enc_out, src_mask)   # (B,1,H)
        context   = context.squeeze(1)                # (B, H)
        alpha_avg = sum(A_list).squeeze(1) / len(A_list)  # (B, S)
        gru_in = torch.cat([emb, context], dim=-1)
        hidden = self.gru(gru_in, hidden)
        pred   = self.fc_out(torch.cat([hidden, context, emb], dim=-1))
        return pred, hidden, alpha_avg, A_list


class Seq2SeqMHA(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, src_mask, teacher_forcing_ratio=0.5):
        B, T = tgt.shape
        enc_out, hidden = self.encoder(src)
        hidden  = hidden.squeeze(0)
        outputs = torch.zeros(B, T, TGT_VOCAB).to(self.device)
        token   = tgt[:, 0]
        for t in range(1, T):
            pred, hidden, _, _ = self.decoder.forward_step(
                token, hidden, enc_out, src_mask)
            outputs[:, t] = pred
            use_tf = random.random() < teacher_forcing_ratio
            token  = tgt[:, t] if use_tf else pred.argmax(-1)
        return outputs

In [ ]:
# ── Build & train Task-5 model ────────────────────────────────────────────────
random.seed(42); np.random.seed(42); torch.manual_seed(42)

enc5   = Encoder(SRC_VOCAB, EMB_DIM, HID_DIM).to(DEVICE)
dec5   = DecoderMHA(TGT_VOCAB, EMB_DIM, HID_DIM, N_HEADS).to(DEVICE)
model5 = Seq2SeqMHA(enc5, dec5, DEVICE).to(DEVICE)

print('Task-5 parameters:', sum(p.numel() for p in model5.parameters()))
train_losses5, val_losses5 = run_training(
    model5, train_loader, val_loader, N_EPOCHS, LR, DEVICE)

In [ ]:
# ── Loss plot ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.plot(range(1, N_EPOCHS+1), train_losses5, 'o-',  label='Train')
plt.plot(range(1, N_EPOCHS+1), val_losses5,   's--', label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Task 5 — Multi-Head Attention: Loss Curves')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── T04 — four per-head heatmaps ─────────────────────────────────────────────
def get_multihead_attn(model, src_tensor, tgt_tensor, mask_tensor, device):
    model.eval()
    with torch.no_grad():
        src  = src_tensor.unsqueeze(0).to(device)
        tgt  = tgt_tensor.unsqueeze(0).to(device)
        mask = mask_tensor.unsqueeze(0).to(device)
        enc_out, hidden = model.encoder(src)
        hidden = hidden.squeeze(0)
        token  = tgt[:, 0]
        head_alphas = [[] for _ in range(N_HEADS)]
        for t in range(1, tgt.shape[1]):
            _, hidden, _, A_list = model.decoder.forward_step(
                token, hidden, enc_out, mask)
            for h_idx, A_h in enumerate(A_list):
                head_alphas[h_idx].append(
                    A_h.squeeze(0).squeeze(0).cpu().numpy())
            token = tgt[:, t]

    src_ids   = src_tensor.tolist()
    src_len   = sum(1 for x in src_ids if x != PAD_IDX)
    src_words = [src_itos[i] for i in src_ids[:src_len]]

    tgt_ids   = tgt_tensor.tolist()
    tgt_words = []
    for i in tgt_ids[1:]:
        w = tgt_itos.get(i, '<unk>')
        if w in ('<pad>', '<eos>'): break
        tgt_words.append(w)
    T_valid = len(tgt_words)

    head_matrices = [
        np.stack(head_alphas[h][:T_valid])[:, :src_len]
        for h in range(N_HEADS)
    ]
    return head_matrices, src_words, tgt_words


idx = 3   # T04
head_mats, sw, tw = get_multihead_attn(
    model5, test_src[idx], test_tgt[idx], test_mask[idx], DEVICE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for h_idx, ax in enumerate(axes.flat):
    mat = head_mats[h_idx]
    im  = ax.imshow(mat, aspect='auto', cmap='Blues')
    ax.set_xticks(range(len(sw)))
    ax.set_xticklabels(sw, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(tw)))
    ax.set_yticklabels(tw, fontsize=8)
    ax.set_title(f'Head {h_idx+1}')
    ax.set_xlabel('Source (EN)'); ax.set_ylabel('Target (DE)')
    plt.colorbar(im, ax=ax)
fig.suptitle('Task 5 — Multi-Head Attention: T04 per-head heatmaps', fontsize=13)
plt.tight_layout(); plt.show()

print('Head attention peaks for T04:')
for h_idx, mat in enumerate(head_mats):
    ti, si = np.unravel_index(mat.argmax(), mat.shape)
    print(f'  Head {h_idx+1}: {tw[ti]} <- {sw[si]}  alpha={mat[ti,si]:.4f}')

print('\nObserved difference between heads:')
print('Some heads concentrate attention sharply on specific content words '
      '(nouns/verbs), producing peaked distributions, while other heads '
      'distribute attention more broadly across multiple source tokens. '
      'This complementary specialisation lets the model capture both '
      'local lexical alignment and longer-range structural dependencies.')

In [ ]:
# ── Greedy decode (Task 5) ────────────────────────────────────────────────────
def greedy_decode_mha(model, src_tensor, mask_tensor, device, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src  = src_tensor.unsqueeze(0).to(device)
        mask = mask_tensor.unsqueeze(0).to(device)
        enc_out, hidden = model.encoder(src)
        hidden = hidden.squeeze(0)
        token  = torch.tensor([SOS_IDX], device=device)
        decoded = []
        for _ in range(max_len):
            pred, hidden, _, _ = model.decoder.forward_step(
                token, hidden, enc_out, mask)
            next_tok = pred.argmax(-1)
            word = tgt_itos.get(next_tok.item(), '<unk>')
            if word == '<eos>': break
            if word not in ('<pad>', '<sos>'):
                decoded.append(word)
            token = next_tok
    return ' '.join(decoded)


print('Task 5 — Greedy translations for T06, T07, T08')
for idx, label in [(5, 'T06'), (6, 'T07'), (7, 'T08')]:
    src_sent, tgt_sent = get_sentence(
        test_src[idx], test_tgt[idx], src_itos, tgt_itos)
    pred = greedy_decode_mha(
        model5, test_src[idx], test_mask[idx], DEVICE)
    print(f'\n{label}')
    print(f'  English source    : {src_sent}')
    print(f'  Ground-truth DE   : {tgt_sent}')
    print(f'  Task-5 prediction : {pred}')

test_loss5 = evaluate(model5, test_loader, criterion, DEVICE)
test_acc5  = token_accuracy(model5, test_loader, DEVICE)
print(f'\nTask-5  Test Loss: {test_loss5:.4f}  Token Accuracy: {test_acc5:.2f}%')

## Task 6: Comparison and Analysis

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────────
results = {
    'Model': [
        'Additive Attention (Task 2)',
        'Scaled Dot-Product Attention (Task 4)',
        'Multi-Head Attention (Task 5)'
    ],
    'Test Loss': [round(test_loss2, 4), round(test_loss4, 4), round(test_loss5, 4)],
    'Token Accuracy (%)': [round(test_acc2, 2), round(test_acc4, 2), round(test_acc5, 2)]
}
df = pd.DataFrame(results)
print(df.to_string(index=False))

In [ ]:
# ── Validation loss comparison plot ──────────────────────────────────────────
epochs = range(1, N_EPOCHS + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, val_losses2, 'o-',  label='Additive Attention (Task 2)')
plt.plot(epochs, val_losses4, 's--', label='Scaled Dot-Product (Task 4)')
plt.plot(epochs, val_losses5, '^:',  label='Multi-Head Attention (Task 5)')
plt.xlabel('Epoch'); plt.ylabel('Validation Loss')
plt.title('Task 6 — Validation Loss Comparison (all three models)')
plt.legend(); plt.tight_layout(); plt.show()